# Playroom Action Annotator — Colab Pipeline

This notebook runs the full YOLO + MMAction2 analysis pipeline on videos stored
in your Google Drive and saves `_results.json` files next to each video.
Those JSON files can then be loaded directly in the desktop UI without re-processing.

**Runtime:** Select `Runtime → Change runtime type → T4 GPU` for best performance.
YOLO and MMAction2 are both significantly faster with a GPU.

---
## Before you start

You need a **GitHub Personal Access Token** so Colab can clone the private repo:

1. On GitHub: click your avatar → **Settings → Developer settings →
   Personal access tokens → Tokens (classic)**
2. Click **Generate new token (classic)**, give it any name, tick **`repo`** scope.
3. Copy the token.
4. In this Colab tab: click the **key icon** (🔑) in the left sidebar → **Secrets**.
5. Add a secret named `GITHUB_TOKEN` and paste your token.
6. Toggle **"Notebook access"** ON for this notebook.


## Cell 1 — Clone / update the repo

Clones the repository on first run.  On subsequent runs it does a `git pull`
to pick up any code changes without re-cloning.

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
repo_url = f'https://{token}@github.com/VictorRyskinB/projectASD.git'

import os
if os.path.isdir('projectASD'):
    print('Repo already cloned — pulling latest changes...')
    !cd projectASD && git pull
else:
    print('Cloning repository...')
    !git clone {repo_url} projectASD 2>/dev/null

%cd projectASD
print('Working directory:', os.getcwd())

## Cell 2 — Install dependencies

Installs the Python packages required by the project.
On Colab (Python 3.10 + GPU), the full MMAction2 stack installs cleanly.

> **Note:** This cell takes 3–5 minutes on first run.
> After installation, the packages are cached for the session.

In [ ]:
# Core dependencies
!pip install -q -r requirements.txt

# MMAction2 stack (works on Colab Python 3.10 + GPU, unlike Windows Python 3.13)
!pip install -q -U openmim
!python -m mim install -q mmengine
!python -m mim install -q mmcv
!python -m mim install -q mmaction2

# Verify GPU
import torch
print(f'\nPyTorch  : {torch.__version__}')
print(f'GPU      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name : {torch.cuda.get_device_name(0)}')

# Verify MMAction2
try:
    import mmaction
    print(f'MMAction2: {mmaction.__version__}')
except ImportError as e:
    print(f'MMAction2 import failed: {e}')

## Cell 3 — Mount Google Drive

Mounts your Google Drive so Colab can read video files and write results JSON files.
A browser pop-up will ask you to sign in and grant access.

**Edit `VIDEO_FOLDER`** to point to the folder in your Drive that contains the `.mp4` files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ← Edit this path to match your Drive folder
VIDEO_FOLDER = '/content/drive/MyDrive/PlayroomVideos'

import os
if not os.path.isdir(VIDEO_FOLDER):
    print(f'WARNING: folder not found: {VIDEO_FOLDER}')
    print('Create the folder in Drive and upload your videos, then re-run this cell.')
else:
    print(f'Video folder: {VIDEO_FOLDER}')
    print(f'Contents: {os.listdir(VIDEO_FOLDER)}')

## Cell 4 — List available videos

Shows every video in your folder and whether a results JSON already exists for it.
Videos marked ✅ have already been processed and can be loaded straight into the desktop UI.

In [ ]:
import os

VIDEO_EXTENSIONS = ('.mp4', '.avi', '.mov', '.mkv', '.wmv')
videos = sorted(
    f for f in os.listdir(VIDEO_FOLDER)
    if f.lower().endswith(VIDEO_EXTENSIONS)
)

if not videos:
    print('No video files found in', VIDEO_FOLDER)
else:
    print(f'Found {len(videos)} video(s):\n')
    for i, v in enumerate(videos):
        stem = v.rsplit('.', 1)[0]
        json_path = os.path.join(VIDEO_FOLDER, stem + '_results.json')
        status = '✅ results exist' if os.path.exists(json_path) else '⏳ not processed'
        print(f'  [{i:2d}]  {v:<40}  {status}')

## Cell 5 — Process a single video

Change `VIDEO_INDEX` to the number shown next to the video you want to process.
You can also tune the `--sample-rate` option:
- `1`  = every frame (most accurate, ~2–4 min per minute of video on GPU)
- `5`  = every 5th frame (~5× faster, still good for multi-second activities)
- `10` = every 10th frame (~10× faster, suitable for rough annotation)

Results are saved as `<videoname>_results.json` in the same Drive folder.

In [ ]:
VIDEO_INDEX = 0   # ← change this to the index shown in Cell 4
SAMPLE_RATE = 5   # ← process every Nth frame (1 = every frame)

video_path  = os.path.join(VIDEO_FOLDER, videos[VIDEO_INDEX])
stem        = videos[VIDEO_INDEX].rsplit('.', 1)[0]
output_path = os.path.join(VIDEO_FOLDER, stem + '_results.json')

print(f'Processing: {videos[VIDEO_INDEX]}')
print(f'Output    : {output_path}\n')

!python main.py --cli \
    --input  "{video_path}" \
    --output "{output_path}" \
    --sample-rate {SAMPLE_RATE}

print('\nDone! The JSON file is saved to your Google Drive.')
print('Open the video in the desktop UI — it will auto-load the results.')

## Cell 6 — Batch process all videos

Processes every video in your folder that does **not** already have a results JSON.
Videos with existing JSON files are skipped automatically.

Useful for overnight runs on a large dataset.

In [ ]:
SAMPLE_RATE = 5   # ← same meaning as in Cell 5

skipped   = []
processed = []
failed    = []

for v in videos:
    stem        = v.rsplit('.', 1)[0]
    video_path  = os.path.join(VIDEO_FOLDER, v)
    output_path = os.path.join(VIDEO_FOLDER, stem + '_results.json')

    if os.path.exists(output_path):
        print(f'  Skipping  {v}  (results already exist)')
        skipped.append(v)
        continue

    print(f'\n━━━ Processing: {v} ━━━')
    ret = os.system(
        f'python main.py --cli '
        f'--input "{video_path}" '
        f'--output "{output_path}" '
        f'--sample-rate {SAMPLE_RATE}'
    )
    if ret == 0:
        processed.append(v)
    else:
        print(f'  ERROR processing {v} (exit code {ret})')
        failed.append(v)

print('\n' + '='*50)
print(f'Batch complete.')
print(f'  Processed : {len(processed)}')
print(f'  Skipped   : {len(skipped)}')
print(f'  Failed    : {len(failed)}')
if failed:
    print(f'  Failed files: {failed}')